In [1]:
# ==========================================
# Projeto de Banco de Dados - Imobiliária
# Baseado no Minimundo e Diagrama ER
# ==========================================

import sqlite3
import pandas as pd

# 1. Conectar/Criar o arquivo de banco de dados SQLite
conn = sqlite3.connect('imobiliaria.db')
cursor = conn.cursor()

# Habilitar restrições de chave estrangeira (Foreign Keys)
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Criação do Esquema Relacional (DDL)
cursor.executescript("""
DROP TABLE IF EXISTS Venda;
DROP TABLE IF EXISTS Compra_do;
DROP TABLE IF EXISTS Contrata_o;
DROP TABLE IF EXISTS Imovel;
DROP TABLE IF EXISTS Corretor;
DROP TABLE IF EXISTS Proprietario;
DROP TABLE IF EXISTS Cliente;

-- Tabela Cliente (Chave Primária: CPF)
CREATE TABLE Cliente (
    cpf TEXT PRIMARY KEY,
    nome TEXT NOT NULL,
    telefone TEXT NOT NULL
);

-- Tabela Proprietário (Chave Primária: CPF)
CREATE TABLE Proprietario (
    cpf TEXT PRIMARY KEY,
    nome TEXT NOT NULL,
    telefone TEXT NOT NULL
);

-- Tabela Corretor (Chave Primária: CRECI)
-- 'avaliador' mapeia se atua ou não como avaliador (0 = Não, 1 = Sim)
CREATE TABLE Corretor (
    creci TEXT PRIMARY KEY,
    nome TEXT NOT NULL,
    telefone TEXT NOT NULL,
    avaliador INTEGER NOT NULL CHECK (avaliador IN (0, 1))
);

-- Tabela Imóvel (Com endereço composto e restrição de tipos)
CREATE TABLE Imovel (
    id_imovel INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo TEXT NOT NULL CHECK (tipo IN ('Casa', 'Apartamento', 'Escritório', 'Comércio')),
    rua TEXT NOT NULL,
    numero INTEGER NOT NULL,
    bairro TEXT NOT NULL,
    cidade TEXT NOT NULL,
    cep TEXT NOT NULL,
    preco REAL NOT NULL,
    cpf_proprietario TEXT NOT NULL,
    FOREIGN KEY (cpf_proprietario) REFERENCES Proprietario(cpf) ON DELETE CASCADE ON UPDATE CASCADE
);

-- Tabela Contrata_o (Proprietário contrata Corretor)
CREATE TABLE Contrata_o (
    cpf_proprietario TEXT NOT NULL,
    creci_corretor TEXT NOT NULL,
    PRIMARY KEY (cpf_proprietario, creci_corretor),
    FOREIGN KEY (cpf_proprietario) REFERENCES Proprietario(cpf) ON DELETE CASCADE ON UPDATE CASCADE,
    FOREIGN KEY (creci_corretor) REFERENCES Corretor(creci) ON DELETE CASCADE ON UPDATE CASCADE
);

-- Tabela Compra_do (Cliente busca/associa-se ao Corretor)
CREATE TABLE Compra_do (
    cpf_cliente TEXT NOT NULL,
    creci_corretor TEXT NOT NULL,
    PRIMARY KEY (cpf_cliente, creci_corretor),
    FOREIGN KEY (cpf_cliente) REFERENCES Cliente(cpf) ON DELETE CASCADE ON UPDATE CASCADE,
    FOREIGN KEY (creci_corretor) REFERENCES Corretor(creci) ON DELETE CASCADE ON UPDATE CASCADE
);

-- Tabela Venda (Representa a negociação intermediada)
-- O campo 'comissao' é um atributo derivado calculado automaticamente
CREATE TABLE Venda (
    id_venda INTEGER PRIMARY KEY AUTOINCREMENT,
    id_imovel INTEGER UNIQUE NOT NULL,
    cpf_cliente TEXT NOT NULL,
    creci_corretor TEXT NOT NULL,
    valor_venda REAL NOT NULL,
    percentual_comissao REAL NOT NULL DEFAULT 5.0,
    comissao REAL GENERATED ALWAYS AS (valor_venda * percentual_comissao / 100.0) STORED,
    FOREIGN KEY (id_imovel) REFERENCES Imovel(id_imovel) ON DELETE RESTRICT ON UPDATE CASCADE,
    FOREIGN KEY (cpf_cliente) REFERENCES Cliente(cpf) ON DELETE RESTRICT ON UPDATE CASCADE,
    FOREIGN KEY (creci_corretor) REFERENCES Corretor(creci) ON DELETE RESTRICT ON UPDATE CASCADE
);
""")

print("Esquema do banco de dados criado com sucesso!")

# 3. Povoamento das Tabelas (DML)
cursor.executescript("""
-- Inserindo Clientes
INSERT INTO Cliente (cpf, nome, telefone) VALUES
('111.111.111-11', 'Daniel Luz', '(84) 99999-1111'),
('222.222.222-22', 'Maria Souza', '(84) 98888-2222');

-- Inserindo Proprietários
INSERT INTO Proprietario (cpf, nome, telefone) VALUES
('333.333.333-33', 'Luiz Feliciano', '(84) 97777-3333'),
('444.444.444-44', 'Klyfithon Paulo', '(84) 96666-4444');

-- Inserindo Corretores (1 = Avaliador, 0 = Não avaliador)
INSERT INTO Corretor (creci, nome, telefone, avaliador) VALUES
('CRECI-1234', 'Carlos Eduardo', '(84) 95555-5555', 1),
('CRECI-5678', 'Fernanda Lima', '(84) 94444-4444', 0);

-- Inserindo Imóveis (Endereço desmembrado conforme a justificativa do PDF)
INSERT INTO Imovel (tipo, rua, numero, bairro, cidade, cep, preco, cpf_proprietario) VALUES
('Apartamento', 'Av. Roberto Freire', 1000, 'Ponta Negra', 'Natal', '59090-000', 350000.00, '333.333.333-33'),
('Casa', 'Rua das Dunas', 150, 'Capim Macio', 'Natal', '59078-000', 500000.00, '444.444.444-44');

-- Inserindo Contratações (Proprietário x Corretor)
INSERT INTO Contrata_o (cpf_proprietario, creci_corretor) VALUES
('333.333.333-33', 'CRECI-1234'),
('444.444.444-44', 'CRECI-5678');

-- Inserindo Atendimentos (Cliente x Corretor)
INSERT INTO Compra_do (cpf_cliente, creci_corretor) VALUES
('111.111.111-11', 'CRECI-1234'),
('222.222.222-22', 'CRECI-5678');

-- Inserindo Vendas (Calcula a comissão automaticamente com base na regra definida)
INSERT INTO Venda (id_imovel, cpf_cliente, creci_corretor, valor_venda, percentual_comissao) VALUES
(1, '111.111.111-11', 'CRECI-1234', 340000.00, 5.0);
""")

conn.commit()
print("Dados inseridos com sucesso no arquivo 'imobiliaria.db'!")

Esquema do banco de dados criado com sucesso!
Dados inseridos com sucesso no arquivo 'imobiliaria.db'!


In [2]:
# Consulta unindo Venda, Cliente, Proprietário, Corretor e Imóvel com Pandas
query = """
SELECT
    v.id_venda,
    c.nome AS cliente,
    p.nome AS proprietario,
    cor.nome AS corretor,
    i.tipo AS tipo_imovel,
    i.bairro,
    v.valor_venda,
    v.percentual_comissao || '%' AS taxa,
    v.comissao
FROM Venda v
JOIN Cliente c ON v.cpf_cliente = c.cpf
JOIN Corretor cor ON v.creci_corretor = cor.creci
JOIN Imovel i ON v.id_imovel = i.id_imovel
JOIN Proprietario p ON i.cpf_proprietario = p.cpf;
"""

df_resultado = pd.read_sql_query(query, conn)
conn.close()

display(df_resultado)

,id_venda,cliente,proprietario,corretor,tipo_imovel,bairro,valor_venda,taxa,comissao
0,1,Daniel Luz,Luiz Feliciano,Carlos Eduardo,Apartamento,Ponta Negra,340000.0,5.0%,17000.0


In [3]:
import sqlite3

# Conecta ao banco de dados existente
conn = sqlite3.connect('imobiliaria.db')
cursor = conn.cursor()

# Habilitar restrições de chave estrangeira
cursor.execute("PRAGMA foreign_keys = ON;")

# Inserindo mais dados (DML Expandido)
cursor.executescript("""
-- Novos Clientes
INSERT OR IGNORE INTO Cliente (cpf, nome, telefone) VALUES
('555.555.555-55', 'Beatriz Medeiros', '(84) 99111-2233'),
('888.888.888-88', 'Lucas Silveira', '(84) 98222-3344');

-- Novos Proprietários
INSERT OR IGNORE INTO Proprietario (cpf, nome, telefone) VALUES
('777.777.777-77', 'Roberta Guedes', '(84) 97333-4455'),
('999.999.999-99', 'Marcos Vinicius', '(84) 96444-5566');

-- Novos Corretores (1 = Avaliador, 0 = Não avaliador)
INSERT OR IGNORE INTO Corretor (creci, nome, telefone, avaliador) VALUES
('CRECI-9988', 'Juliana Azevedo', '(84) 95555-6677', 1);

-- Novos Imóveis (Tipos permitidos: 'Casa', 'Apartamento', 'Escritório', 'Comércio')
INSERT INTO Imovel (tipo, rua, numero, bairro, cidade, cep, preco, cpf_proprietario) VALUES
('Casa', 'Rua Engenheiro Roberto Freire', 450, 'Petrópolis', 'Natal', '59012-000', 680000.00, '777.777.777-77'),
('Escritório', 'Av. Afonso Pena', 700, 'Tirol', 'Natal', '59020-100', 290000.00, '999.999.999-99'),
('Comércio', 'Rua dos Eucaliptos', 12, 'Candelária', 'Natal', '59066-120', 850000.00, '777.777.777-77');

-- Novas Contratações (Proprietário x Corretor)
INSERT OR IGNORE INTO Contrata_o (cpf_proprietario, creci_corretor) VALUES
('777.777.777-77', 'CRECI-9988'),
('999.999.999-99', 'CRECI-1234');

-- Novos Atendimentos/Associações (Cliente x Corretor)
INSERT OR IGNORE INTO Compra_do (cpf_cliente, creci_corretor) VALUES
('555.555.555-55', 'CRECI-9988'),
('888.888.888-88', 'CRECI-9988');

-- Novas Vendas (Lembrando que id_imovel em Venda possui restrição UNIQUE, então usamos imóveis novos: 2, 3 e 4)
INSERT INTO Venda (id_imovel, cpf_cliente, creci_corretor, valor_venda, percentual_comissao) VALUES
(2, '555.555.555-55', 'CRECI-9988', 650000.00, 6.0),
(3, '888.888.888-88', 'CRECI-9988', 280000.00, 5.0);
""")

conn.commit()
conn.close()

print("Novos dados inseridos com sucesso!")

Novos dados inseridos com sucesso!


In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('imobiliaria.db')

query = """
SELECT
    v.id_venda,
    c.nome AS cliente,
    p.nome AS proprietario,
    cor.nome AS corretor,
    i.tipo AS tipo_imovel,
    i.bairro,
    v.valor_venda,
    v.percentual_comissao || '%' AS taxa,
    v.comissao
FROM Venda v
JOIN Cliente c ON v.cpf_cliente = c.cpf
JOIN Corretor cor ON v.creci_corretor = cor.creci
JOIN Imovel i ON v.id_imovel = i.id_imovel
JOIN Proprietario p ON i.cpf_proprietario = p.cpf;
"""

df_resultado = pd.read_sql_query(query, conn)
conn.close()

display(df_resultado)

,id_venda,cliente,proprietario,corretor,tipo_imovel,bairro,valor_venda,taxa,comissao
0,1,Daniel Luz,Luiz Feliciano,Carlos Eduardo,Apartamento,Ponta Negra,340000.0,5.0%,17000.0
1,2,Beatriz Medeiros,Klyfithon Paulo,Juliana Azevedo,Casa,Capim Macio,650000.0,6.0%,39000.0
2,3,Lucas Silveira,Roberta Guedes,Juliana Azevedo,Casa,Petrópolis,280000.0,5.0%,14000.0


In [12]:
import sqlite3
import pandas as pd

# 1. Conecta ao banco de dados existente
conn = sqlite3.connect('imobiliaria.db')
cursor = conn.cursor()

# Habilitar restrições de chave estrangeira
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Inserção de dados expandidos e diversificados
cursor.executescript("""
-- Clientes Adicionais
INSERT OR IGNORE INTO Cliente (cpf, nome, telefone) VALUES
('111.111.111-11', 'Daniel Luz', '(84) 99999-1111'),
('222.222.222-22', 'Maria Souza', '(84) 98888-2222'),
('555.555.555-55', 'Beatriz Medeiros', '(84) 99111-2233'),
('666.666.666-66', 'Lucas Silveira', '(84) 98222-3344'),
('777.777.777-77', 'Camila Albuquerque', '(84) 97333-5566'),
('888.888.888-88', 'Thiago Bezerra', '(84) 96444-7788');

-- Proprietários Adicionais
INSERT OR IGNORE INTO Proprietario (cpf, nome, telefone) VALUES
('333.333.333-33', 'Luiz Feliciano', '(84) 97777-3333'),
('444.444.444-44', 'Klyfithon Paulo', '(84) 96666-4444'),
('123.456.789-00', 'Roberto Guedes', '(84) 99888-1122'),
('234.567.890-11', 'Marcos Vinicius', '(84) 98777-2233'),
('345.678.901-22', 'Juliana Paiva', '(84) 97666-3344');

-- Corretores Adicionais (1 = Avaliador, 0 = Não avaliador)
INSERT OR IGNORE INTO Corretor (creci, nome, telefone, avaliador) VALUES
('CRECI-1234', 'Carlos Eduardo', '(84) 95555-5555', 1),
('CRECI-5678', 'Fernanda Lima', '(84) 94444-4444', 0),
('CRECI-9988', 'Juliana Azevedo', '(84) 95555-6677', 1),
('CRECI-4321', 'Marcos Paulo', '(84) 94444-3322', 1),
('CRECI-8765', 'Priscila Duarte', '(84) 93333-1100', 0);

-- Imóveis Adicionais (Tipos aceitos: 'Casa', 'Apartamento', 'Escritório', 'Comércio')
INSERT INTO Imovel (tipo, rua, numero, bairro, cidade, cep, preco, cpf_proprietario) VALUES
('Apartamento', 'Av. Roberto Freire', 1000, 'Ponta Negra', 'Natal', '59090-000', 350000.00, '333.333.333-33'),
('Casa', 'Rua das Dunas', 150, 'Capim Macio', 'Natal', '59078-000', 500000.00, '444.444.444-44'),
('Casa', 'Rua Engenheiro Roberto Freire', 450, 'Petrópolis', 'Natal', '59012-000', 680000.00, '123.456.789-00'),
('Escritório', 'Av. Afonso Pena', 700, 'Tirol', 'Natal', '59020-100', 290000.00, '234.567.890-11'),
('Comércio', 'Rua dos Eucaliptos', 12, 'Candelária', 'Natal', '59066-120', 850000.00, '345.678.901-22'),
('Apartamento', 'Av. Hermes da Fonseca', 1200, 'Tirol', 'Natal', '59020-555', 420000.00, '123.456.789-00'),
('Casa', 'Rua Praia de Genipabu', 85, 'Ponta Negra', 'Natal', '59092-100', 530000.00, '234.567.890-11');

-- Contratações (Proprietário x Corretor)
INSERT OR IGNORE INTO Contrata_o (cpf_proprietario, creci_corretor) VALUES
('333.333.333-33', 'CRECI-1234'),
('444.444.444-44', 'CRECI-5678'),
('123.456.789-00', 'CRECI-9988'),
('234.567.890-11', 'CRECI-4321'),
('345.678.901-22', 'CRECI-8765');

-- Atendimentos / Associações (Cliente x Corretor)
INSERT OR IGNORE INTO Compra_do (cpf_cliente, creci_corretor) VALUES
('111.111.111-11', 'CRECI-1234'),
('222.222.222-22', 'CRECI-5678'),
('555.555.555-55', 'CRECI-9988'),
('666.666.666-66', 'CRECI-4321'),
('777.777.777-77', 'CRECI-8765'),
('888.888.888-88', 'CRECI-9988');

-- Vendas (Garantindo IDs de imóveis únicos que ainda não foram vendidos)
INSERT OR IGNORE INTO Venda (id_venda, id_imovel, cpf_cliente, creci_corretor, valor_venda, percentual_comissao) VALUES
(1, 1, '111.111.111-11', 'CRECI-1234', 340000.00, 5.0),
(2, 3, '555.555.555-55', 'CRECI-9988', 650000.00, 6.0),
(3, 4, '666.666.666-66', 'CRECI-4321', 280000.00, 5.0),
(4, 5, '777.777.777-77', 'CRECI-8765', 830000.00, 5.5),
(5, 6, '888.888.888-88', 'CRECI-9988', 410000.00, 5.0);
""")

conn.commit()
print("Banco de dados populado com dados robustos com sucesso!")

# 3. Consulta de validação utilizando Pandas
query = """
SELECT
    v.id_venda,
    c.nome AS cliente,
    p.nome AS proprietario,
    cor.nome AS corretor,
    i.tipo AS tipo_imovel,
    i.bairro,
    v.valor_venda,
    v.percentual_comissao || '%' AS taxa,
    v.comissao
FROM Venda v
JOIN Cliente c ON v.cpf_cliente = c.cpf
JOIN Corretor cor ON v.creci_corretor = cor.creci
JOIN Imovel i ON v.id_imovel = i.id_imovel
JOIN Proprietario p ON i.cpf_proprietario = p.cpf;
"""

df_resultado = pd.read_sql_query(query, conn)
conn.close()

display(df_resultado)

Banco de dados populado com dados robustos com sucesso!


,id_venda,cliente,proprietario,corretor,tipo_imovel,bairro,valor_venda,taxa,comissao
0,1,Daniel Luz,Luiz Feliciano,Carlos Eduardo,Apartamento,Ponta Negra,340000.0,5.0%,17000.0
1,2,Beatriz Medeiros,Klyfithon Paulo,Juliana Azevedo,Casa,Capim Macio,650000.0,6.0%,39000.0
2,3,Lucas Silveira,Roberta Guedes,Juliana Azevedo,Casa,Petrópolis,280000.0,5.0%,14000.0
3,5,Lucas Silveira,Roberto Guedes,Juliana Azevedo,Casa,Petrópolis,410000.0,5.0%,20500.0
4,6,Camila Albuquerque,Marcos Vinicius,Priscila Duarte,Escritório,Tirol,830000.0,5.5%,45650.0
5,7,Lucas Silveira,Roberta Guedes,Juliana Azevedo,Comércio,Candelária,410000.0,5.0%,20500.0


In [13]:
import sqlite3
import pandas as pd

# Conecta ao banco
conn = sqlite3.connect('imobiliaria.db')

query_vendas = """
SELECT
    v.id_venda,
    c.nome AS cliente,
    p.nome AS proprietario,
    cor.nome AS corretor,
    i.tipo AS tipo_imovel,
    i.bairro,
    v.valor_venda,
    v.percentual_comissao || '%' AS taxa,
    v.comissao
FROM Venda v
JOIN Cliente c ON v.cpf_cliente = c.cpf
JOIN Corretor cor ON v.creci_corretor = cor.creci
JOIN Imovel i ON v.id_imovel = i.id_imovel
JOIN Proprietario p ON i.cpf_proprietario = p.cpf;
"""

df_vendas = pd.read_sql_query(query_vendas, conn)
conn.close()

print("--- RELATÓRIO DE VENDAS ---")
display(df_vendas)

--- RELATÓRIO DE VENDAS ---


,id_venda,cliente,proprietario,corretor,tipo_imovel,bairro,valor_venda,taxa,comissao
0,1,Daniel Luz,Luiz Feliciano,Carlos Eduardo,Apartamento,Ponta Negra,340000.0,5.0%,17000.0
1,2,Beatriz Medeiros,Klyfithon Paulo,Juliana Azevedo,Casa,Capim Macio,650000.0,6.0%,39000.0
2,3,Lucas Silveira,Roberta Guedes,Juliana Azevedo,Casa,Petrópolis,280000.0,5.0%,14000.0
3,5,Lucas Silveira,Roberto Guedes,Juliana Azevedo,Casa,Petrópolis,410000.0,5.0%,20500.0
4,6,Camila Albuquerque,Marcos Vinicius,Priscila Duarte,Escritório,Tirol,830000.0,5.5%,45650.0
5,7,Lucas Silveira,Roberta Guedes,Juliana Azevedo,Comércio,Candelária,410000.0,5.0%,20500.0


In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('imobiliaria.db')

query_imoveis = """
SELECT
    i.id_imovel,
    i.tipo,
    i.rua,
    i.numero,
    i.bairro,
    i.cidade,
    i.preco,
    pr.nome AS proprietario,
    pr.telefone AS tel_proprietario
FROM Imovel i
JOIN Proprietario pr ON i.cpf_proprietario = pr.cpf;
"""

df_imoveis = pd.read_sql_query(query_imoveis, conn)
conn.close()

print("--- LISTA DE IMÓVEIS ---")
display(df_imoveis)

--- LISTA DE IMÓVEIS ---


,id_imovel,tipo,rua,numero,bairro,cidade,preco,proprietario,tel_proprietario
0,1,Apartamento,Av. Roberto Freire,1000,Ponta Negra,Natal,350000.0,Luiz Feliciano,(84) 97777-3333
1,2,Casa,Rua das Dunas,150,Capim Macio,Natal,500000.0,Klyfithon Paulo,(84) 96666-4444
2,3,Casa,Rua Engenheiro Roberto Freire,450,Petrópolis,Natal,680000.0,Roberta Guedes,(84) 97333-4455
3,4,Escritório,Av. Afonso Pena,700,Tirol,Natal,290000.0,Marcos Vinicius,(84) 96444-5566
4,5,Comércio,Rua dos Eucaliptos,12,Candelária,Natal,850000.0,Roberta Guedes,(84) 97333-4455
5,6,Casa,Rua Engenheiro Roberto Freire,450,Petrópolis,Natal,680000.0,Roberto Guedes,(84) 99888-1122
6,7,Escritório,Av. Afonso Pena,700,Tirol,Natal,290000.0,Marcos Vinicius,(84) 98777-2233
7,8,Comércio,Rua dos Eucaliptos,12,Candelária,Natal,850000.0,Juliana Paiva,(84) 97666-3344
8,9,Apartamento,Av. Hermes da Fonseca,1200,Tirol,Natal,420000.0,Roberto Guedes,(84) 99888-1122
9,10,Casa,Rua Praia de Genipabu,85,Pontas dos Secos,Natal,530000.0,Marcos Vinicius,(84) 98777-2233


In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('imobiliaria.db')

query_corretores = """
SELECT
    creci,
    nome,
    telefone,
    CASE
        WHEN avaliador = 1 THEN 'Sim'
        ELSE 'Não'
    END AS e_avaliador
FROM Corretor;
"""

df_corretores = pd.read_sql_query(query_corretores, conn)
conn.close()

print("--- LISTA DE CORRETORES ---")
display(df_corretores)

--- LISTA DE CORRETORES ---


,creci,nome,telefone,e_avaliador
0,CRECI-1234,Carlos Eduardo,(84) 95555-5555,Sim
1,CRECI-5678,Fernanda Lima,(84) 94444-4444,Não
2,CRECI-9988,Juliana Azevedo,(84) 95555-6677,Sim
3,CRECI-4321,Marcos Paulo,(84) 94444-3322,Sim
4,CRECI-8765,Priscila Duarte,(84) 93333-1100,Não


In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('imobiliaria.db')

# Troque 'Cliente' pelo nome de qualquer outra tabela (Proprietario, Corretor, Imovel, etc.)
df_tabela = pd.read_sql_query("SELECT * FROM Cliente", conn)
conn.close()

display(df_tabela)

,cpf,nome,telefone
0,111.111.111-11,Daniel Luz,(84) 99999-1111
1,222.222.222-22,Maria Souza,(84) 98888-2222
2,555.555.555-55,Beatriz Medeiros,(84) 99111-2233
3,888.888.888-88,Lucas Silveira,(84) 98222-3344
4,666.666.666-66,Lucas Silveira,(84) 98222-3344
5,777.777.777-77,Camila Albuquerque,(84) 97333-5566
